# Mel + Chroma + Tempo

## Tanítás

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    Activation, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Concatenate, Dense, Dropout, Lambda
)
from tensorflow.keras.models import Model

# --- 1. BEMENETEK ---
input_mel    = Input(shape=(128, 1280, 1), name='mel_input')
input_chroma = Input(shape=(12,  1280, 1), name='chroma_input')
input_tempo  = Input(shape=(384, 1280, 1), name='tempo_input')

# --- 2. BRANCH ÉPÍTŐ FÜGGVÉNY ---
def create_branch(x, filters, pool_sizes, branch_name):
    for i, f in enumerate(filters):
        x = Conv2D(f, kernel_size=(3, 3), padding='same',
                   name=f'{branch_name}_conv_{i+1}')(x)
        x = BatchNormalization(name=f'{branch_name}_bn_{i+1}')(x)
        x = Activation('relu', name=f'{branch_name}_relu_{i+1}')(x)
        x = MaxPooling2D(pool_size=pool_sizes[i],
                         name=f'{branch_name}_pool_{i+1}')(x)

    avg_pool = GlobalAveragePooling2D(name=f'{branch_name}_global_avg')(x)
    max_pool = GlobalMaxPooling2D   (name=f'{branch_name}_global_max')(x)

    return Concatenate(name=f'{branch_name}_pool_concat')([avg_pool, max_pool])

# --- 3. ÁGAK ---

# MEL ÁG: (2,4) pooling az időtengely gyorsabb zsugorítása miatt
branch_mel = create_branch(
    input_mel,
    filters    = [32, 64, 128, 256],
    pool_sizes = [(2,4), (2,4), (2,2), (2,2)],   # időtengely: 1280→320→80→40→20
    branch_name = "mel"
)

# TEMPOGRAM ÁG: ugyanolyan stratégia mint a mel
branch_tempo = create_branch(
    input_tempo,
    filters    = [32, 64, 128, 256],
    pool_sizes = [(2,4), (2,4), (2,2), (2,2)],
    branch_name = "tempo"
)

# CHROMA ÁG: csak vízszintes pooling (12 hangmagasság-osztály védelme)
branch_chroma = create_branch(
    input_chroma,
    filters    = [16, 32, 64],
    pool_sizes = [(1,4), (1,4), (1,4)],           # időtengely: 1280→320→80→20
    branch_name = "chroma"
)

# --- 4. ÖSSZEFÉSÜLÉS ---
merged = Concatenate(name='concat_all_branches')(
    [branch_mel, branch_chroma, branch_tempo]
)

# --- 5. DÖNTÉSHOZÓ RÉTEGEK (BatchNorm hozzáadva) ---
z = Dense(1024, activation='relu', name='dense_1')(merged)
z = BatchNormalization(name='bn_dense_1')(z)
z = Dropout(0.4, name='dropout_1')(z)

z = Dense(512, activation='relu', name='dense_2')(z)
z = BatchNormalization(name='bn_dense_2')(z)
z = Dropout(0.4, name='dropout_2')(z)

# --- 6. KIMENET ---
# L2 normalizálás: egységnyi hosszú vektorok → cosine metrikához helyes
output_raw = Dense(400, activation='linear', name='word2vec_output')(z)
output_layer = Lambda(
    lambda x: tf.math.l2_normalize(x, axis=1),
    name='l2_norm'
)(output_raw)

# --- 7. LOSS: 1 - cosine_similarity (Keras minimalizál, ezért kell a negálás) ---
def cosine_loss(y_true, y_pred):
    # cosine_similarity() értéke [-1, 1], negálva minimalizálható
    return 1.0 - tf.reduce_mean(
        tf.reduce_sum(
            tf.math.l2_normalize(y_true, axis=1) * y_pred,
            axis=1
        )
    )

# --- 8. ÖSSZESZEREÉS ÉS FORDÍTÁS ---
model = Model(
    inputs  = [input_mel, input_chroma, input_tempo],
    outputs = output_layer
)

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss      = cosine_loss,
    metrics   = ['mae']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mel_input           │ (None, 128, 1280, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tempo_input         │ (None, 384, 1280, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_conv_1 (Conv2D) │ (None, 128, 1280, │        320 │ mel_input[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tempo_conv_1        │ (None, 384, 1280, │        320 │ tempo_input[0][0] │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_bn_1            │ (None, 128, 1280, │        128 │ mel_conv_1[0][0]  │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tempo_bn_1          │ (None, 384, 1280, │        128 │ tempo_conv_1[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_relu_1          │ (None, 128, 1280, │          0 │ mel_bn_1[0][0]    │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tempo_relu_1        │ (None, 384, 1280, │          0 │ tempo_bn_1[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_pool_1          │ (None, 64, 320,   │          0 │ mel_relu_1[0][0]  │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ chroma_input        │ (None, 12, 1280,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tempo_pool_1        │ (None, 192, 320,  │          0 │ tempo_relu_1[0][… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_conv_2 (Conv2D) │ (None, 64, 320,   │     18,496 │ mel_pool_1[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ chroma_conv_1       │ (None, 12, 1280,  │        160 │ chroma_input[0][… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tempo_conv_2        │ (None, 192, 320,  │     18,496 │ tempo_pool_1[0][… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_bn_2            │ (None, 64, 320,   │        256 │ mel_conv_2[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ chroma_bn_1         │ (None, 12, 1280,  │         64 │ chroma_conv_1[0]… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tempo_bn_2          │ (None, 192, 320,  │        256 │ tempo_conv_2[0][

 Total params: 2,720,080 (10.38 MB)

 Trainable params: 2,714,864 (10.36 MB)

 Non-trainable params: 5,216 (20.38 KB)

In [ ]:
import os

import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    Activation, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Concatenate, Dense, Dropout
)
from tensorflow.keras.models import Model
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from gensim.models import Word2Vec

# --- ÚTVONALAK A COLABON ---
H5_PATH = "/content/spotify_dataset_compressed.h5"
W2V_PATH = "/content/song2vec.model" # Frissítve a te fájlnevedre!
SAVE_PATH = "/content/drive/MyDrive/spotify_cnn_model.keras"

# ==========================================
# 1. A PINCÉR: DATA GENERATOR
# ==========================================
class SpotifyDataGenerator(Sequence):
    def __init__(self, h5_path, w2v_model, split_name='train', batch_size=64, shuffle=True):
        self.h5_path = h5_path
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.w2v_model = w2v_model

        with h5py.File(self.h5_path, 'r') as hf:
            splits = hf['ml/split'][:]
            splits_str = np.array([s.decode('utf-8') if isinstance(s, bytes) else s for s in splits])
            self.indices = np.where(splits_str == split_name)[0]

        print(f"✅ Generátor '{split_name}': {len(self.indices)} dal (Batch size: {batch_size}).")
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        batch_indices = np.sort(batch_indices)

        with h5py.File(self.h5_path, 'r') as hf:
            X_mel = hf['spectrograms/mel'][batch_indices]
            X_chroma = hf['spectrograms/chroma'][batch_indices]
            X_tempo = hf['spectrograms/tempogram'][batch_indices]
            uris = hf['ml/track_uri'][batch_indices]

        # Csatorna dimenzió hozzáadása
        X_mel = np.expand_dims(X_mel, axis=-1)
        X_chroma = np.expand_dims(X_chroma, axis=-1)
        X_tempo = np.expand_dims(X_tempo, axis=-1)

        y_batch = np.zeros((len(batch_indices), 400), dtype=np.float32)
        for i, uri_bytes in enumerate(uris):
            uri = uri_bytes.decode('utf-8') if isinstance(uri_bytes, bytes) else uri_bytes
            if uri in self.w2v_model.wv:
                y_batch[i] = self.w2v_model.wv[uri]

        # A JAVÍTÁS: List helyett Tuple-t adunk vissza a bemeneteknél
        # Eredeti: return [X_mel, X_chroma, X_tempo], y_batch
        return (X_mel, X_chroma, X_tempo), y_batch

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

# ==========================================
# 2. AZ AGY: MODELL ARCHITEKTÚRA
# ==========================================
def create_branch(x, filters, pool_sizes, branch_name):
    for i, f in enumerate(filters):
        x = Conv2D(f, kernel_size=(3, 3), padding='same', name=f'{branch_name}_conv_{i+1}')(x)
        x = BatchNormalization(name=f'{branch_name}_bn_{i+1}')(x)
        x = Activation('relu', name=f'{branch_name}_relu_{i+1}')(x)
        x = MaxPooling2D(pool_size=pool_sizes[i], name=f'{branch_name}_pool_{i+1}')(x)
    avg_pool = GlobalAveragePooling2D(name=f'{branch_name}_global_avg')(x)
    max_pool = GlobalMaxPooling2D(name=f'{branch_name}_global_max')(x)
    return Concatenate(name=f'{branch_name}_pool_concat')([avg_pool, max_pool])

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

print("\n🚀 Modell építése...")
input_mel    = Input(shape=(128, 1280, 1), name='mel_input')
input_chroma = Input(shape=(12,  1280, 1), name='chroma_input')
input_tempo  = Input(shape=(384, 1280, 1), name='tempo_input')

branch_mel = create_branch(input_mel, [32, 64, 128, 256], [(2,4), (2,4), (2,2), (2,2)], "mel")
branch_tempo = create_branch(input_tempo, [32, 64, 128, 256], [(2,4), (2,4), (2,2), (2,2)], "tempo")
branch_chroma = create_branch(input_chroma, [16, 32, 64], [(1,4), (1,4), (1,4)], "chroma")

merged = Concatenate(name='concat_all_branches')([branch_mel, branch_chroma, branch_tempo])
z = Dense(1024, activation='relu', name='dense_1')(merged)
z = BatchNormalization(name='bn_dense_1')(z)
z = Dropout(0.4, name='dropout_1')(z)
z = Dense(512, activation='relu', name='dense_2')(z)
z = BatchNormalization(name='bn_dense_2')(z)
z = Dropout(0.4, name='dropout_2')(z)

output_raw = Dense(400, activation='linear', name='word2vec_output')(z)
output_layer = L2NormLayer(name='l2_norm')(output_raw)

model = Model(inputs=[input_mel, input_chroma, input_tempo], outputs=output_layer)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss=cosine_loss, metrics=['mae'])

# ==========================================
# 3. A STARTPISZTOLY: TANÍTÁS
# ==========================================
print("\n📚 Word2Vec betöltése...")
w2v_model = Word2Vec.load(W2V_PATH)

print("\n⚙️ Generátorok inicializálása...")
train_gen = SpotifyDataGenerator(H5_PATH, w2v_model, split_name='train', batch_size=64)
val_gen = SpotifyDataGenerator(H5_PATH, w2v_model, split_name='val', batch_size=64, shuffle=False)

# --- SÚLYOK BETÖLTÉSE (BIZTONSÁGI MÓDSZER) ---
if os.path.exists(SAVE_PATH):
    print(f"\n💾 Korábbi állapot megtalálva!")
    model = tf.keras.models.load_model(
        SAVE_PATH,
        custom_objects={
            'cosine_loss': cosine_loss,
            'L2NormLayer': L2NormLayer
        }
    )
    print("✅ Modell sikeresen betöltve!")
else:
    print(f"\n⚠️ FIGYELEM: Nem található fájl a {SAVE_PATH} útvonalon!")
    print("Indulás a nulláról.")

callbacks = [
    # A legjobb modellt mindig egyből kimenti a Google Drive-odra!
    ModelCheckpoint(SAVE_PATH, save_best_only=True, monitor='val_loss', mode='min', verbose=1),
    # Ha 5 körön át nem javul, leállítja magát
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
]

print("\n🔥 GPU TANÍTÁS FOLYTATÁSA...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    initial_epoch=8,    # Megmondjuk neki, hogy a 7. már kész van!
    callbacks=callbacks
)


🚀 Modell építése...

📚 Word2Vec betöltése...

⚙️ Generátorok inicializálása...
✅ Generátor 'train': 21642 dal (Batch size: 64).
✅ Generátor 'val': 2705 dal (Batch size: 64).

💾 Korábbi állapot megtalálva!
✅ Modell sikeresen betöltve!

🔥 GPU TANÍTÁS FOLYTATÁSA...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 9/20
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - loss: 0.3626 - mae: 0.1471
Epoch 9: val_loss improved from None to 0.38442, saving model to /content/drive/MyDrive/spotify_cnn_model.keras

Epoch 9: finished saving model to /content/drive/MyDrive/spotify_cnn_model.keras
339/339 ━━━━━━━━━━━━━━━━━━━━ 2969s 9s/step - loss: 0.3622 - mae: 0.1469 - val_loss: 0.3844 - val_mae: 0.1478
Epoch 10/20
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - loss: 0.3595 - mae: 0.1467
Epoch 10: val_loss improved from 0.38442 to 0.37731, saving model to /content/drive/MyDrive/spotify_cnn_model.keras

Epoch 10: finished saving model to /content/drive/MyDrive/spotify_cnn_model.keras
339/339 ━━━━━━━━━━━━━━━━━━━━ 2879s 8s/step - loss: 0.3599 - mae: 0.1468 - val_loss: 0.3773 - val_mae: 0.1475
Epoch 11/20
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - loss: 0.3582 - mae: 0.1469
Epoch 11: val_loss did not improve from 0.37731
339/339 ━━━━━━━━━━━━━━━━━━━━ 2899s 9s/step - loss: 0.3574 - mae: 0.1467 - val_loss: 0.3879 - val_ma

## Kiértékelés

### Releváns uri, word2vec térben keresés (megkapom, mennyire jók a CNN vektorok)

In [2]:
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm 
import random
from sklearn.metrics import roc_auc_score

# ==========================================
# 1. ÚTVONALAK ÉS BEÁLLÍTÁSOK
# ==========================================
H5_PATH = "../Dataset/spotify_dataset.h5"
W2V_PATH = "../Models/song2vec.model"
CNN_PATH = "../Models/spotify_cnn_model.keras" 
K_VALUES = [1, 5, 10, 20, 50, 100, 200, 500] 

# Tesztelési korlátok a gyorsasághoz
NUM_TEST_SAMPLES = 1000  # None, ha az összeset akarod
STEP_SIZE = 2          # Minden hanyadik teszt elemet vegyük

# ==========================================
# 2. SEGÉDFÜGGVÉNYEK ÉS EGYEDI RÉTEGEK
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    y_true_norm = tf.math.l2_normalize(y_true, axis=1)
    y_pred_norm = tf.math.l2_normalize(y_pred, axis=1)
    return 1.0 - tf.reduce_mean(tf.reduce_sum(y_true_norm * y_pred_norm, axis=1))

def calculate_metrics(recommended_uris, relevant_uris, K):
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

# --- AUC SZÁMÍTÁSA (Sampled ROC-AUC) ---
def calculate_auc(query_vector, relevant_uris, w2v_model, num_negatives=1000):
    """
    Megméri, hogy a modell mekkora eséllyel rangsorolja előrébb a 
    valóban releváns dalokat a véletlen negatívokhoz képest.
    """
    # 1. Pozitív minták pontszámai (amikkel már volt közös listán)
    pos_scores = []
    for uri in relevant_uris:
        if uri in w2v_model.wv:
            # Koszinusz hasonlóság (vektorok normáltak, így a dot product elég)
            score = np.dot(query_vector, w2v_model.wv[uri])
            pos_scores.append(score)
    
    if not pos_scores: 
        return 0.5 # Ha nincs pozitív adat, a véletlen szintet adjuk vissza
    
    # 2. Negatív minták (véletlen dalok, amikkel SOHA nem volt közös listán)
    neg_scores = []
    all_uris = list(w2v_model.wv.key_to_index.keys())
    
    # Biztonsági korlát a mintavételhez
    n_neg = min(num_negatives, len(all_uris) - len(relevant_uris))
    
    while len(neg_scores) < n_neg:
        random_uri = random.choice(all_uris)
        if random_uri not in relevant_uris:
            score = np.dot(query_vector, w2v_model.wv[random_uri])
            neg_scores.append(score)
            
    # 3. ROC-AUC kiszámítása a scikit-learn segítségével
    y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
    y_scores = pos_scores + neg_scores
    
    try:
        return roc_auc_score(y_true, y_scores)
    except:
        return 0.5

# ==========================================
# 3. FŐPROGRAM
# ==========================================
def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    cnn_model = tf.keras.models.load_model(
        CNN_PATH, custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

    print("\n2. Kapcsolati háló (Ground Truth) építése a memóriában...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)
    
    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            uri = uris[i].decode('utf-8') if isinstance(uris[i], bytes) else uris[i]
            
            track_to_playlists[uri].add(pid)
            playlist_to_tracks[pid].add(uri)

    print("\n3. Teszt halmaz előkészítése...")
    with h5py.File(H5_PATH, "r") as hf:
        splits = hf["ml/split"][:]
        all_test_indices = [i for i, s in enumerate(splits) if (s.decode('utf-8') if isinstance(s, bytes) else s) == "test"]
        
        # Determinisztikus mintavétel
        test_indices = all_test_indices[::STEP_SIZE]
        
        if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
            test_indices = test_indices[:NUM_TEST_SAMPLES]
            
        print(f"Tesztelésre kijelölve: {len(test_indices)} dal.")

        # Eredménytároló szótár
        results = {
            "baseline": {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
            "cnn":      {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}
        }

        print("\n4. Kiértékelés indítása (AUC + Top-K)...")
        for idx in tqdm(test_indices, desc="Dalok tesztelése"):
            uri = hf["ml/track_uri"][idx]
            uri = uri.decode('utf-8') if isinstance(uri, bytes) else uri
            
            if uri not in w2v_model.wv:
                continue
                
            # --- RELEVÁNS DALOK (Akikkel már volt egy listán) ---
            pids_of_track = track_to_playlists.get(uri, set())
            if not pids_of_track: continue 
                
            relevant_uris = set()
            for pid in pids_of_track:
                relevant_uris.update(playlist_to_tracks[pid])
            
            relevant_uris.discard(uri) # Önmagát ne ajánlja
            relevant_uris = {u for u in relevant_uris if u in w2v_model.wv} # Csak ami benne van a szótárban
            
            if not relevant_uris: continue

            # --- A) BASELINE (Word2Vec) ---
            baseline_vector = w2v_model.wv[uri]
            w2v_similars = w2v_model.wv.most_similar(uri, topn=max(K_VALUES))
            w2v_recs = [u for u, score in w2v_similars]

            # --- B) CNN MODELL (Audio) ---
            mel = np.expand_dims(np.expand_dims(hf["spectrograms/mel"][idx], axis=0), axis=-1)
            chroma = np.expand_dims(np.expand_dims(hf["spectrograms/chroma"][idx], axis=0), axis=-1)
            tempo = np.expand_dims(np.expand_dims(hf["spectrograms/tempogram"][idx], axis=0), axis=-1)
            
            pred_vector = cnn_model.predict([mel, chroma, tempo], verbose=0)[0]
            
            cnn_similars = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
            cnn_recs = [u for u, score in cnn_similars if u != uri][:max(K_VALUES)]

            # --- AUC SZÁMÍTÁS ---
            results["baseline"]["auc"].append(calculate_auc(baseline_vector, relevant_uris, w2v_model))
            results["cnn"]["auc"].append(calculate_auc(pred_vector, relevant_uris, w2v_model))

            # --- TOP-K METRIKÁK SZÁMÍTÁSA ---
            for k in K_VALUES:
                # Baseline
                hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
                results["baseline"][k]["hr"].append(hr_b)
                results["baseline"][k]["prec"].append(prec_b)
                results["baseline"][k]["rec"].append(rec_b)
                
                # CNN
                hr_c, prec_c, rec_c = calculate_metrics(cnn_recs, relevant_uris, k)
                results["cnn"][k]["hr"].append(hr_c)
                results["cnn"][k]["prec"].append(prec_c)
                results["cnn"][k]["rec"].append(rec_c)

    # ==========================================
    # 4. EREDMÉNYEK ÖSSZESÍTÉSE
    # ==========================================
    print("\n" + "="*60)
    print("VÉGLEGES KIÉRTÉKELÉS (Item-to-Item Similarity)")
    print("="*60)
    
    print(f"\nGLOBÁLIS RANGSOROLÁSI MINŐSÉG:")
    print(f"  BASELINE (Word2Vec) AUC: {np.mean(results['baseline']['auc'])*100:.2f}%")
    print(f"  CNN MODEL (Audio)    AUC: {np.mean(results['cnn']['auc'])*100:.2f}%")

    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print(f"  BASELINE -> HR: {np.mean(results['baseline'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['baseline'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['baseline'][k]['rec'])*100:.2f}%")
        print(f"  CNN MODELL -> HR: {np.mean(results['cnn'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['cnn'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['cnn'][k]['rec'])*100:.2f}%")
    print("\n" + "="*60)

if __name__ == "__main__":
    main()

1. Modellek betöltése...


2. Kapcsolati háló (Ground Truth) építése a memóriában...


Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [02:19<00:00, 476549.24it/s]



3. Teszt halmaz előkészítése...
Tesztelésre kijelölve: 1000 dal.

4. Kiértékelés indítása (AUC + Top-K)...


Dalok tesztelése: 100%|██████████| 1000/1000 [06:36<00:00,  2.52it/s]



VÉGLEGES KIÉRTÉKELÉS (Item-to-Item Similarity)

GLOBÁLIS RANGSOROLÁSI MINŐSÉG:
  BASELINE (Word2Vec) AUC: 82.42%
  CNN MODEL (Audio)    AUC: 73.66%

--- TOP 1 AJÁNLÁS ---
  BASELINE -> HR: 91.30% | Prec: 91.30% | Rec: 0.02%
  CNN MODELL -> HR: 17.00% | Prec: 17.00% | Rec: 0.00%

--- TOP 5 AJÁNLÁS ---
  BASELINE -> HR: 97.50% | Prec: 86.94% | Rec: 0.07%
  CNN MODELL -> HR: 40.80% | Prec: 17.96% | Rec: 0.01%

--- TOP 10 AJÁNLÁS ---
  BASELINE -> HR: 98.80% | Prec: 84.41% | Rec: 0.13%
  CNN MODELL -> HR: 52.70% | Prec: 18.27% | Rec: 0.01%

--- TOP 20 AJÁNLÁS ---
  BASELINE -> HR: 99.30% | Prec: 80.93% | Rec: 0.24%
  CNN MODELL -> HR: 64.50% | Prec: 18.57% | Rec: 0.03%

--- TOP 50 AJÁNLÁS ---
  BASELINE -> HR: 99.80% | Prec: 75.46% | Rec: 0.53%
  CNN MODELL -> HR: 76.20% | Prec: 18.50% | Rec: 0.07%

--- TOP 100 AJÁNLÁS ---
  BASELINE -> HR: 99.80% | Prec: 70.46% | Rec: 0.93%
  CNN MODELL -> HR: 82.20% | Prec: 18.32% | Rec: 0.14%

--- TOP 200 AJÁNLÁS ---
  BASELINE -> HR: 99.90% | Prec: 64

### Releváns uri, hibrid téren (valós környezet)

In [1]:
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm 
import random
from sklearn.metrics import roc_auc_score
import copy  # Hozzáadva a hibrid modell klónozásához!

# ==========================================
# 1. ÚTVONALAK ÉS BEÁLLÍTÁSOK
# ==========================================
H5_PATH = "../Dataset/spotify_dataset.h5"
W2V_PATH = "../Models/song2vec.model"
CNN_PATH = "../Models/spotify_cnn_model.keras" 
HYBRID_MATRIX_PATH = "../Models/hybrid_embedding_matrix.npy" # A létrehozott hibrid mátrixod

K_VALUES = [1, 5, 10, 20, 50, 100, 200, 500] 

# Tesztelési korlátok a gyorsasághoz
NUM_TEST_SAMPLES = 1000  # None, ha az összeset akarod
STEP_SIZE = 2          # Minden hanyadik teszt elemet vegyük

# ==========================================
# 2. SEGÉDFÜGGVÉNYEK ÉS EGYEDI RÉTEGEK
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    y_true_norm = tf.math.l2_normalize(y_true, axis=1)
    y_pred_norm = tf.math.l2_normalize(y_pred, axis=1)
    return 1.0 - tf.reduce_mean(tf.reduce_sum(y_true_norm * y_pred_norm, axis=1))

def calculate_metrics(recommended_uris, relevant_uris, K):
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

# --- AUC SZÁMÍTÁSA (Sampled ROC-AUC) ---
def calculate_auc(query_vector, relevant_uris, w2v_model, num_negatives=1000):
    """
    Megméri, hogy a modell mekkora eséllyel rangsorolja előrébb a 
    valóban releváns dalokat a véletlen negatívokhoz képest.
    """
    # 1. Pozitív minták pontszámai (amikkel már volt közös listán)
    pos_scores = []
    for uri in relevant_uris:
        if uri in w2v_model.wv:
            # Koszinusz hasonlóság (vektorok normáltak, így a dot product elég)
            score = np.dot(query_vector, w2v_model.wv[uri])
            pos_scores.append(score)
    
    if not pos_scores: 
        return 0.5 # Ha nincs pozitív adat, a véletlen szintet adjuk vissza
    
    # 2. Negatív minták (véletlen dalok, amikkel SOHA nem volt közös listán)
    neg_scores = []
    all_uris = list(w2v_model.wv.key_to_index.keys())
    
    # Biztonsági korlát a mintavételhez
    n_neg = min(num_negatives, len(all_uris) - len(relevant_uris))
    
    while len(neg_scores) < n_neg:
        random_uri = random.choice(all_uris)
        if random_uri not in relevant_uris:
            score = np.dot(query_vector, w2v_model.wv[random_uri])
            neg_scores.append(score)
            
    # 3. ROC-AUC kiszámítása a scikit-learn segítségével
    y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
    y_scores = pos_scores + neg_scores
    
    try:
        return roc_auc_score(y_true, y_scores)
    except:
        return 0.5

# ==========================================
# 3. FŐPROGRAM
# ==========================================
def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    cnn_model = tf.keras.models.load_model(
        CNN_PATH, custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

    print("\n[+] HIBRID vektortér betöltése a memóriába...")
    # Klónozzuk a w2v modellt, hogy a szótár (vocab) megmaradjon
    hybrid_w2v = copy.deepcopy(w2v_model)
    
    # Betöltjük a numpy mátrixot
    hybrid_matrix = np.load(HYBRID_MATRIX_PATH)
    
    # FELÜLÍRJUK a klónozott modell vektorait a hibrid mátrixszal!
    hybrid_w2v.wv.vectors = hybrid_matrix
    
    # KRITIKUS LÉPÉS: Újraszámoljuk a normákat a cosine similarity-hez
    hybrid_w2v.wv.fill_norms(force=True) 
    print("    Hibrid vektortér sikeresen integrálva!")

    print("\n2. Kapcsolati háló (Ground Truth) építése a memóriában...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)
    
    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            uri = uris[i].decode('utf-8') if isinstance(uris[i], bytes) else uris[i]
            
            track_to_playlists[uri].add(pid)
            playlist_to_tracks[pid].add(uri)

    print("\n3. Teszt halmaz előkészítése...")
    with h5py.File(H5_PATH, "r") as hf:
        splits = hf["ml/split"][:]
        all_test_indices = [i for i, s in enumerate(splits) if (s.decode('utf-8') if isinstance(s, bytes) else s) == "test"]
        
        # Determinisztikus mintavétel
        test_indices = all_test_indices[::STEP_SIZE]
        
        if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
            test_indices = test_indices[:NUM_TEST_SAMPLES]
            
        print(f"Tesztelésre kijelölve: {len(test_indices)} dal.")

        # Eredménytároló szótár (kibővítve a hibriddel)
        results = {
            "baseline": {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
            "cnn":      {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
            "hybrid":   {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}
        }

        print("\n4. Kiértékelés indítása (AUC + Top-K)...")
        for idx in tqdm(test_indices, desc="Dalok tesztelése"):
            uri = hf["ml/track_uri"][idx]
            uri = uri.decode('utf-8') if isinstance(uri, bytes) else uri
            
            if uri not in w2v_model.wv:
                continue
                
            # --- RELEVÁNS DALOK (Akikkel már volt egy listán) ---
            pids_of_track = track_to_playlists.get(uri, set())
            if not pids_of_track: continue 
                
            relevant_uris = set()
            for pid in pids_of_track:
                relevant_uris.update(playlist_to_tracks[pid])
            
            relevant_uris.discard(uri) # Önmagát ne ajánlja
            relevant_uris = {u for u in relevant_uris if u in w2v_model.wv} # Csak ami benne van a szótárban
            
            if not relevant_uris: continue

            # --- A) BASELINE (Word2Vec) ---
            baseline_vector = w2v_model.wv[uri]
            w2v_similars = w2v_model.wv.most_similar(uri, topn=max(K_VALUES))
            w2v_recs = [u for u, score in w2v_similars]

            # --- B) CNN MODELL (Audio) ---
            mel = np.expand_dims(np.expand_dims(hf["spectrograms/mel"][idx], axis=0), axis=-1)
            chroma = np.expand_dims(np.expand_dims(hf["spectrograms/chroma"][idx], axis=0), axis=-1)
            tempo = np.expand_dims(np.expand_dims(hf["spectrograms/tempogram"][idx], axis=0), axis=-1)
            
            pred_vector = cnn_model.predict([mel, chroma, tempo], verbose=0)[0]
            
            cnn_similars = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
            cnn_recs = [u for u, score in cnn_similars if u != uri][:max(K_VALUES)]

            # --- C) HIBRID MODELL ---
            hybrid_similars = hybrid_w2v.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
            hybrid_recs = [u for u, score in hybrid_similars if u != uri][:max(K_VALUES)]

            # --- AUC SZÁMÍTÁS ---
            results["baseline"]["auc"].append(calculate_auc(baseline_vector, relevant_uris, w2v_model))
            results["cnn"]["auc"].append(calculate_auc(pred_vector, relevant_uris, w2v_model))
            results["hybrid"]["auc"].append(calculate_auc(pred_vector, relevant_uris, hybrid_w2v))

            # --- TOP-K METRIKÁK SZÁMÍTÁSA ---
            for k in K_VALUES:
                # Baseline
                hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
                results["baseline"][k]["hr"].append(hr_b)
                results["baseline"][k]["prec"].append(prec_b)
                results["baseline"][k]["rec"].append(rec_b)
                
                # CNN
                hr_c, prec_c, rec_c = calculate_metrics(cnn_recs, relevant_uris, k)
                results["cnn"][k]["hr"].append(hr_c)
                results["cnn"][k]["prec"].append(prec_c)
                results["cnn"][k]["rec"].append(rec_c)
                
                # Hybrid
                hr_h, prec_h, rec_h = calculate_metrics(hybrid_recs, relevant_uris, k)
                results["hybrid"][k]["hr"].append(hr_h)
                results["hybrid"][k]["prec"].append(prec_h)
                results["hybrid"][k]["rec"].append(rec_h)

    # ==========================================
    # 4. EREDMÉNYEK ÖSSZESÍTÉSE
    # ==========================================
    print("\n" + "="*70)
    print("🏆 VÉGLEGES KIÉRTÉKELÉS (Item-to-Item Similarity) 🏆")
    print("="*70)
    
    print(f"\n✨ GLOBÁLIS RANGSOROLÁSI MINŐSÉG:")
    print(f"  BASELINE (Word2Vec) AUC: {np.mean(results['baseline']['auc'])*100:.2f}%")
    print(f"  CNN MODEL (Audio)   AUC: {np.mean(results['cnn']['auc'])*100:.2f}%")
    print(f"  HIBRID MODELL       AUC: {np.mean(results['hybrid']['auc'])*100:.2f}%")

    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print(f"  BASELINE -> HR: {np.mean(results['baseline'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['baseline'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['baseline'][k]['rec'])*100:.2f}%")
        print(f"  CNN      -> HR: {np.mean(results['cnn'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['cnn'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['cnn'][k]['rec'])*100:.2f}%")
        print(f"  HIBRID   -> HR: {np.mean(results['hybrid'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['hybrid'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['hybrid'][k]['rec'])*100:.2f}%")
    print("\n" + "="*70)

if __name__ == "__main__":
    main()

1. Modellek betöltése...


[+] HIBRID vektortér betöltése a memóriába...
    Hibrid vektortér sikeresen integrálva!

2. Kapcsolati háló (Ground Truth) építése a memóriában...


Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [02:21<00:00, 469478.11it/s]



3. Teszt halmaz előkészítése...
Tesztelésre kijelölve: 1000 dal.

4. Kiértékelés indítása (AUC + Top-K)...


Dalok tesztelése: 100%|██████████| 1000/1000 [08:57<00:00,  1.86it/s]



🏆 VÉGLEGES KIÉRTÉKELÉS (Item-to-Item Similarity) 🏆

✨ GLOBÁLIS RANGSOROLÁSI MINŐSÉG:
  BASELINE (Word2Vec) AUC: 82.45%
  CNN MODEL (Audio)   AUC: 73.65%
  HIBRID MODELL       AUC: 57.67%

--- TOP 1 AJÁNLÁS ---
  BASELINE -> HR: 91.30% | Prec: 91.30% | Rec: 0.02%
  CNN      -> HR: 17.00% | Prec: 17.00% | Rec: 0.00%
  HIBRID   -> HR: 54.40% | Prec: 54.40% | Rec: 0.01%

--- TOP 5 AJÁNLÁS ---
  BASELINE -> HR: 97.50% | Prec: 86.94% | Rec: 0.07%
  CNN      -> HR: 40.80% | Prec: 17.96% | Rec: 0.01%
  HIBRID   -> HR: 86.30% | Prec: 56.48% | Rec: 0.03%

--- TOP 10 AJÁNLÁS ---
  BASELINE -> HR: 98.80% | Prec: 84.41% | Rec: 0.13%
  CNN      -> HR: 52.70% | Prec: 18.27% | Rec: 0.01%
  HIBRID   -> HR: 91.90% | Prec: 55.84% | Rec: 0.06%

--- TOP 20 AJÁNLÁS ---
  BASELINE -> HR: 99.30% | Prec: 80.93% | Rec: 0.24%
  CNN      -> HR: 64.50% | Prec: 18.57% | Rec: 0.03%
  HIBRID   -> HR: 95.60% | Prec: 55.54% | Rec: 0.12%

--- TOP 50 AJÁNLÁS ---
  BASELINE -> HR: 99.80% | Prec: 75.46% | Rec: 0.53%
  CNN

### Szekvenciális hibrid téren

In [4]:
import numpy as np
import faiss
from tqdm import tqdm
import math

# --- KONFIGURÁCIÓ ---
HYBRID_MATRIX_PATH = "../Models/hybrid_embedding_matrix.npy"
TEST_PIDS_PATH = "../Models/test_pids.npy"

# Az új K értékek listája
K_VALUES = [1, 5, 10, 20, 50, 100, 200, 500] 
MAX_K = max(K_VALUES) # A FAISS kereséshez csak a legnagyobbra lesz szükségünk

def calculate_ndcg(rank, k):
    """NDCG kiszámítása (mivel 1 releváns elemünk van, az IDCG = 1)"""
    if rank < k:
        return 1.0 / math.log2(rank + 2) # rank+2 mert 0-tól indexelünk
    return 0

def calculate_map(rank, k):
    """MAP kiszámítása (1 releváns elem esetén = 1/(rank+1))"""
    if rank < k:
        return 1.0 / (rank + 1)
    return 0

# 1. Adatok betöltése
print("Adatok betöltése...")
hybrid_matrix = np.load(HYBRID_MATRIX_PATH).astype('float32')
test_playlists = np.load(TEST_PIDS_PATH, allow_pickle=True)

# Normalizálás a koszinusz-hasonlósághoz
print("Vektorok normalizálása...")
faiss.normalize_L2(hybrid_matrix)

# 2. FAISS Index felépítése
print("FAISS index építése...")
d = hybrid_matrix.shape[1]
index = faiss.IndexFlatIP(d) 
index.add(hybrid_matrix)

# 3. Kiértékelés előkészítése
# Egy szótárat (dictionary) hozunk létre, ami minden K értékhez tárolja a metrikákat
results = {k: {"hr_count": 0, "map_sum": 0.0, "ndcg_sum": 0.0} for k in K_VALUES}
total_playlists = 0

print(f"Kiértékelés futtatása {len(test_playlists):,} listán...")

for pl in tqdm(test_playlists[::10]): # Maradt a minden 10. lista a gyorsasághoz
    if len(pl) < 2:
        continue
    
    # Utolsó elem a cél, előtte lévők a kontextus
    context_indices = pl[:-1]
    target_idx = pl[-1]
    
    # --- SÚLYOZOTT CENTROID ---
    vectors = hybrid_matrix[context_indices]
    weights = np.arange(1, len(context_indices) + 1).astype('float32').reshape(-1, 1)
    
    weighted_centroid = np.sum(vectors * weights, axis=0) / np.sum(weights)
    
    # Keresés előtti normalizálás
    centroid_norm = weighted_centroid / (np.linalg.norm(weighted_centroid) + 1e-8)
    centroid_norm = centroid_norm.reshape(1, -1).astype('float32')

    # --- KERESÉS (FAISS) ---
    # Most a MAX_K-ra (500) + a kontextus hosszára keresünk
    D, I = index.search(centroid_norm, MAX_K + len(context_indices))
    
    # Ajánlások szűrése (ne ajánljuk azt, ami már a listában van)
    recommendations = []
    for idx in I[0]:
        if idx not in context_indices:
            recommendations.append(idx)
        if len(recommendations) == MAX_K:
            break

    # --- METRIKÁK SZÁMÍTÁSA MINDEN K-RA ---
    total_playlists += 1
    
    if target_idx in recommendations:
        # Megkeressük, hányadik helyen van a találat a max 500-as listában
        rank = recommendations.index(target_idx)
        
        # Végigmegyünk a K értékeken, és frissítjük azokat, amikbe belefér a találat
        for k in K_VALUES:
            if rank < k:
                results[k]["hr_count"] += 1
                results[k]["map_sum"] += calculate_map(rank, k)
                results[k]["ndcg_sum"] += calculate_ndcg(rank, k)

# 4. Eredmények összesítése és kiíratása
print("\n" + "="*50)
print("VÉGLEGES KIÉRTÉKELÉS (Next-Item Prediction)")
print("="*50)

for k in K_VALUES:
    final_hr = results[k]["hr_count"] / total_playlists
    final_map = results[k]["map_sum"] / total_playlists
    final_ndcg = results[k]["ndcg_sum"] / total_playlists
    
    print(f"\n--- TOP {k} AJÁNLÁS ---")
    print(f"  Hit Rate (HR@{k}):  {final_hr*100:.2f}%")
    print(f"  MAP@{k}:            {final_map*100:.4f}%")
    print(f"  NDCG@{k}:           {final_ndcg*100:.4f}%")

print("\n" + "="*50)

Adatok betöltése...
Vektorok normalizálása...
FAISS index építése...
Kiértékelés futtatása 99,180 listán...


100%|██████████| 9918/9918 [06:10<00:00, 26.75it/s]


VÉGLEGES KIÉRTÉKELÉS (Next-Item Prediction)

--- TOP 1 AJÁNLÁS ---
  Hit Rate (HR@1):  0.27%
  MAP@1:            0.2722%
  NDCG@1:           0.2722%

--- TOP 5 AJÁNLÁS ---
  Hit Rate (HR@5):  0.68%
  MAP@5:            0.3986%
  NDCG@5:           0.4663%

--- TOP 10 AJÁNLÁS ---
  Hit Rate (HR@10):  1.00%
  MAP@10:            0.4391%
  NDCG@10:           0.5681%

--- TOP 20 AJÁNLÁS ---
  Hit Rate (HR@20):  1.61%
  MAP@20:            0.4803%
  NDCG@20:           0.7218%

--- TOP 50 AJÁNLÁS ---
  Hit Rate (HR@50):  3.22%
  MAP@50:            0.5293%
  NDCG@50:           1.0362%

--- TOP 100 AJÁNLÁS ---
  Hit Rate (HR@100):  5.21%
  MAP@100:            0.5575%
  NDCG@100:           1.3592%

--- TOP 200 AJÁNLÁS ---
  Hit Rate (HR@200):  8.81%
  MAP@200:            0.5829%
  NDCG@200:           1.8608%

--- TOP 500 AJÁNLÁS ---
  Hit Rate (HR@500):  15.39%
  MAP@500:            0.6034%
  NDCG@500:           2.6469%



### Szekvenciális zárt, tartalomalapú téren

In [2]:
import numpy as np
import faiss
from tqdm import tqdm
import math

# --- KONFIGURÁCIÓ ---
HYBRID_MATRIX_PATH = "../Models/ae_vectors_closed_world.npy"
TEST_PIDS_PATH = "../Models/test_pids_ae.npy"

# Az új K értékek listája
K_VALUES = [1, 5, 10, 20, 50, 100, 200, 500] 
MAX_K = max(K_VALUES) # A FAISS kereséshez csak a legnagyobbra lesz szükségünk

def calculate_ndcg(rank, k):
    """NDCG kiszámítása (mivel 1 releváns elemünk van, az IDCG = 1)"""
    if rank < k:
        return 1.0 / math.log2(rank + 2) # rank+2 mert 0-tól indexelünk
    return 0

def calculate_map(rank, k):
    """MAP kiszámítása (1 releváns elem esetén = 1/(rank+1))"""
    if rank < k:
        return 1.0 / (rank + 1)
    return 0

# 1. Adatok betöltése
print("Adatok betöltése...")
hybrid_matrix = np.load(HYBRID_MATRIX_PATH).astype('float32')
test_playlists = np.load(TEST_PIDS_PATH, allow_pickle=True)

# Normalizálás a koszinusz-hasonlósághoz
print("Vektorok normalizálása...")
faiss.normalize_L2(hybrid_matrix)

# 2. FAISS Index felépítése
print("FAISS index építése...")
d = hybrid_matrix.shape[1]
index = faiss.IndexFlatIP(d) 
index.add(hybrid_matrix)

# 3. Kiértékelés előkészítése
# Egy szótárat (dictionary) hozunk létre, ami minden K értékhez tárolja a metrikákat
results = {k: {"hr_count": 0, "map_sum": 0.0, "ndcg_sum": 0.0} for k in K_VALUES}
total_playlists = 0

print(f"Kiértékelés futtatása {len(test_playlists):,} listán...")

for pl in tqdm(test_playlists): # Maradt a minden 10. lista a gyorsasághoz
    if len(pl) < 2:
        continue
    
    # Utolsó elem a cél, előtte lévők a kontextus
    context_indices = [idx - 1 for idx in pl[:-1]]
    target_idx = pl[-1] - 1
    
    # --- SÚLYOZOTT CENTROID ---
    vectors = hybrid_matrix[context_indices]
    weights = np.arange(1, len(context_indices) + 1).astype('float32').reshape(-1, 1)
    
    weighted_centroid = np.sum(vectors * weights, axis=0) / np.sum(weights)
    
    # Keresés előtti normalizálás
    centroid_norm = weighted_centroid / (np.linalg.norm(weighted_centroid) + 1e-8)
    centroid_norm = centroid_norm.reshape(1, -1).astype('float32')

    # --- KERESÉS (FAISS) ---
    # Most a MAX_K-ra (500) + a kontextus hosszára keresünk
    D, I = index.search(centroid_norm, MAX_K + len(context_indices))
    
    # Ajánlások szűrése (ne ajánljuk azt, ami már a listában van)
    recommendations = []
    for idx in I[0]:
        if idx not in context_indices:
            recommendations.append(idx)
        if len(recommendations) == MAX_K:
            break

    # --- METRIKÁK SZÁMÍTÁSA MINDEN K-RA ---
    total_playlists += 1
    
    if target_idx in recommendations:
        # Megkeressük, hányadik helyen van a találat a max 500-as listában
        rank = recommendations.index(target_idx)
        
        # Végigmegyünk a K értékeken, és frissítjük azokat, amikbe belefér a találat
        for k in K_VALUES:
            if rank < k:
                results[k]["hr_count"] += 1
                results[k]["map_sum"] += calculate_map(rank, k)
                results[k]["ndcg_sum"] += calculate_ndcg(rank, k)

# 4. Eredmények összesítése és kiíratása
print("\n" + "="*50)
print("VÉGLEGES KIÉRTÉKELÉS (Next-Item Prediction)")
print("="*50)

for k in K_VALUES:
    final_hr = results[k]["hr_count"] / total_playlists
    final_map = results[k]["map_sum"] / total_playlists
    final_ndcg = results[k]["ndcg_sum"] / total_playlists
    
    print(f"\n--- TOP {k} AJÁNLÁS ---")
    print(f"  Hit Rate (HR@{k}):  {final_hr*100:.2f}%")
    print(f"  MAP@{k}:            {final_map*100:.4f}%")
    print(f"  NDCG@{k}:           {final_ndcg*100:.4f}%")

print("\n" + "="*50)

Adatok betöltése...
Vektorok normalizálása...
FAISS index építése...
Kiértékelés futtatása 88,316 listán...


100%|██████████| 88316/88316 [04:35<00:00, 320.66it/s]


VÉGLEGES KIÉRTÉKELÉS (Next-Item Prediction)

--- TOP 1 AJÁNLÁS ---
  Hit Rate (HR@1):  0.02%
  MAP@1:            0.0249%
  NDCG@1:           0.0249%

--- TOP 5 AJÁNLÁS ---
  Hit Rate (HR@5):  0.09%
  MAP@5:            0.0471%
  NDCG@5:           0.0578%

--- TOP 10 AJÁNLÁS ---
  Hit Rate (HR@10):  0.19%
  MAP@10:            0.0603%
  NDCG@10:           0.0904%

--- TOP 20 AJÁNLÁS ---
  Hit Rate (HR@20):  0.34%
  MAP@20:            0.0705%
  NDCG@20:           0.1282%

--- TOP 50 AJÁNLÁS ---
  Hit Rate (HR@50):  0.78%
  MAP@50:            0.0838%
  NDCG@50:           0.2137%

--- TOP 100 AJÁNLÁS ---
  Hit Rate (HR@100):  1.43%
  MAP@100:            0.0929%
  NDCG@100:           0.3182%

--- TOP 200 AJÁNLÁS ---
  Hit Rate (HR@200):  2.51%
  MAP@200:            0.1004%
  NDCG@200:           0.4686%

--- TOP 500 AJÁNLÁS ---
  Hit Rate (HR@500):  5.42%
  MAP@500:            0.1093%
  NDCG@500:           0.8155%

